<a href="https://colab.research.google.com/github/raheelarif86/AI_Training_November25/blob/main/OT_Cybersecurity_Risk_Assessment_09b_January_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# OT Nuclear Risk Assessment – Optimized Version
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2 requests

import os
import io
import json
import re
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
import requests
from functools import lru_cache
from google.colab import userdata
from openai import OpenAI

# ============================================================
# 0) CONFIG & FLAGS
# ============================================================

# Models
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# Data paths (adjust if needed)
COMPLIANCE_PATH = "data/FANR-REG-08_V2.pdf"
HEATMAP_PATH = "data/Nuclear_OT_Risk_Heatmap.xlsx"
CONTROL_PATH = "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx"

# Performance / behavior flags
DEBUG_LOG = False   # If True, prints key timing/info to console (minimal)
TEST_MODE = False   # If True, you can mock / simplify external behavior

# Likelihood scale
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

# Impact scale
IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

# --------------------------------------------------
# 1) Configure OpenAI client using Colab Secrets
# --------------------------------------------------

os.environ["OPENAI_API_KEY"] = userdata.get("openai")

def safe_get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

NVD_API_KEY = safe_get_secret("nvd_api_key")
VULNCHECK_API_KEY = safe_get_secret("vulncheck_api_key")

if NVD_API_KEY is None:
    print("⚠️ NVD_API_KEY not found — using unauthenticated NVD mode (rate-limited).")

if VULNCHECK_API_KEY is None:
    print("⚠️ VULNCHECK_API_KEY not found — VulnCheck historical exploitation disabled.")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

# --------------------------------------------------
# Global OT-nuclear system prompt
# --------------------------------------------------

NUCLEAR_OT_SYSTEM_PROMPT = """
You are an expert in nuclear Operational Technology (OT) cybersecurity.

Your answers MUST be grounded in:
- NEI 08-09 (Cyber Security Plan for Nuclear Power Reactors),
- NRC RG 5.71,
- FANR-REG-08,
- ISA/IEC 62443 as adapted for nuclear facilities,
- safety-critical engineering principles for nuclear plants.

STRICT REQUIREMENTS:
- Focus on OT systems (control systems, safety systems, engineering workstations, process networks).
- Emphasize deterministic system behavior, safety-first design, and defense-in-depth for physical processes.
- Account for legacy systems, limited patching windows, maintenance constraints, and physical process coupling.
- Align with regulatory expectations and nuclear safety culture.
- Avoid generic enterprise IT controls and tools (e.g., SIEM, EDR, CASB, DLP, cloud-first architectures) unless explicitly and clearly justified as applicable to nuclear OT.
- Do NOT recommend cloud-based solutions for critical OT functions.
- Prefer engineering, procedural, and physical controls over purely IT-centric technical controls.
"""

# ============================================================
# 2) Utility: Embeddings + RAG (Optimized with caching)
# ============================================================

EMBED_CACHE = {}

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Newlines removed to avoid embedding inconsistencies.
    """
    text = text.replace("\n", " ")
    if TEST_MODE:
        # Deterministic simple embedding for test mode
        return [hash(text) % 997 / 997.0] * 16

    # Use cache to avoid recomputing embeddings
    key = ("emb", EMBEDDING_MODEL, text)
    if key in EMBED_CACHE:
        return EMBED_CACHE[key]
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    emb = resp.data[0].embedding
    EMBED_CACHE[key] = emb
    return emb

def cosine_similarity(a, b):
    """
    Compute cosine similarity between two embedding vectors.
    """
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """
    Read text from multiple file formats:
    txt, csv, json, pdf, docx.
    Returns raw text for RAG indexing.
    """
    if not os.path.exists(path):
        if DEBUG_LOG:
            print(f"[DEBUG] File not found for RAG: {path}")
        return ""

    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

@lru_cache(maxsize=16)
def build_rag_index(path: str):
    """
    Build a simple RAG index:
    - Load file text
    - Chunk into ~700-word segments
    - Embed each chunk

    Cached via lru_cache so built once per path.
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    if DEBUG_LOG:
        print(f"[DEBUG] RAG index built for {path}, chunks={len(index)}")
    return index

def rag_search(index, query, top_k=3):
    """
    Retrieve top-k most relevant chunks using cosine similarity.
    """
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent (Optimized)
# ============================================================

def clamp_likelihood(v):
    """
    Normalize model output to one of the allowed likelihood labels.
    """
    if v is None:
        return "possible"
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    """
    Ask the LLM to classify likelihood based on title + causes.
    Used as a fallback when web data is unavailable.
    """
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Explain your reasoning in 1–2 short sentences in a nuclear OT context, then output ONLY the label on the last line.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content.splitlines()[-1])

def extract_cve(text):
    """
    Extract the first CVE identifier if present in the text.
    Example: CVE-2023-12345
    """
    match = re.search(r"CVE-\d{4}-\d{4,7}", text, flags=re.IGNORECASE)
    return match.group(0).upper() if match else None

@lru_cache(maxsize=512)
def fetch_nvd_data(cve_id):
    """
    Fetch vulnerability data from NVD API.
    Uses API key if available. Cached per CVE.
    """
    if TEST_MODE:
        return {"exploitability": 2.5, "impact": 3.0, "vector": "TEST/AV:N/..."}
    url = f"https://services.nvd.nist.gov/rest/json/cve/2.0?cveId={cve_id}"
    headers = {}
    if NVD_API_KEY:
        headers["apiKey"] = NVD_API_KEY

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return None
        cve = vulns[0].get("cve", {})
        metrics = cve.get("metrics", {})
        cvss_list = metrics.get("cvssMetricV31") or metrics.get("cvssMetricV30") or metrics.get("cvssMetricV2")
        if not cvss_list:
            return None
        m = cvss_list[0]
        exploitability = m.get("exploitabilityScore")
        impact = m.get("impactScore")
        vector = m.get("cvssData", {}).get("vectorString")
        return {
            "exploitability": exploitability,
            "impact": impact,
            "vector": vector
        }
    except Exception:
        return None

def map_exploitability_to_likelihood(score):
    """
    Map CVSS exploitability score (0.0–3.9) to qualitative likelihood.
    """
    if score is None:
        return None
    try:
        s = float(score)
    except ValueError:
        return None

    if s <= 0.5:
        return "very unlikely"
    if s <= 1.5:
        return "unlikely"
    if s <= 2.5:
        return "possible"
    if s <= 3.2:
        return "likely"
    return "very likely"

@lru_cache(maxsize=512)
def fetch_epss(cve_id):
    """
    Fetch EPSS (Exploit Prediction Scoring System) probability from FIRST.org API.
    Cached per CVE.
    """
    if TEST_MODE:
        return 0.35
    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        items = data.get("data", [])
        if not items:
            return None
        epss_str = items[0].get("epss")
        if epss_str is None:
            return None
        return float(epss_str)
    except Exception:
        return None

def map_epss_to_likelihood(epss):
    """
    Map EPSS probability (0–1) to qualitative likelihood for 'exposure'.
    """
    if epss is None:
        return None
    if epss < 0.05:
        return "very unlikely"
    if epss < 0.20:
        return "unlikely"
    if epss < 0.50:
        return "possible"
    if epss < 0.75:
        return "likely"
    return "very likely"

# ============================================================
# Historical Occurrence via VulnCheck API
# ============================================================

@lru_cache(maxsize=512)
def fetch_historical_occurrence(cve_id):
    """
    Fetch historical exploitation likelihood using VulnCheck API.
    Converts VulnCheck exploitation intelligence into a 0–1 probability.
    """
    if not VULNCHECK_API_KEY or not cve_id:
        return None

    if TEST_MODE:
        # Optional: simple deterministic value in test mode
        return 0.25

    url = f"https://api.vulncheck.com/v3/cve/{cve_id}"
    headers = {"Authorization": f"Bearer {VULNCHECK_API_KEY}"}

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None

        data = resp.json()

        # VulnCheck fields we can use
        exploited = data.get("exploited", False)
        kev = data.get("kev", False)
        exploit_poc = data.get("exploit_poc", False)
        trending = data.get("trending", False)
        malware = data.get("malware", False)
        ransomware = data.get("ransomware", False)

        # Convert to probability (0–1)
        score = 0.0
        if exploited: score += 0.50
        if kev: score += 0.20
        if exploit_poc: score += 0.10
        if trending: score += 0.10
        if malware: score += 0.05
        if ransomware: score += 0.05

        # Clamp to 1.0
        score = min(score, 1.0)

        return score

    except Exception:
        return None


def map_history_prob_to_likelihood(p):
    """
    Map historical probability (0–1) to qualitative likelihood.
    """
    if p is None:
        return None
    if p < 0.05:
        return "very unlikely"
    if p < 0.20:
        return "unlikely"
    if p < 0.50:
        return "possible"
    if p < 0.75:
        return "likely"
    return "very likely"

def likelihood_agent(threat_actor, exposure_input, title, causes):
    """
    Multi-factor likelihood agent with caching and minimized external calls.
    """

    # Normalize TAC from user
    tac_label = clamp_likelihood(threat_actor)

    # Combine title + causes to search for CVE IDs
    cve = extract_cve(f"{title} {causes}")

    vuln_label = None
    exp_label = None
    hist_label = None

    # Web-enriched paths if CVE is present
    if cve:
        nvd = fetch_nvd_data(cve)
        if nvd and nvd.get("exploitability") is not None:
            vuln_label = map_exploitability_to_likelihood(nvd["exploitability"])

        epss = fetch_epss(cve)
        if epss is not None:
            exp_label = map_epss_to_likelihood(epss)

        # VulnCheck-based historical exploitation likelihood
        hist_prob = fetch_historical_occurrence(cve)
        if hist_prob is not None:
            hist_label = map_history_prob_to_likelihood(hist_prob)

    # Fallbacks if web data is not available
    if vuln_label is None:
        vuln_label = infer_likelihood_from_model(title + " (vuln)", causes)

    if exp_label is None:
        exp_label = clamp_likelihood(exposure_input)

    if hist_label is None:
        hist_label = infer_likelihood_from_model(title + " (history)", causes)

    # Convert to numeric scores 0–4
    tac_score = L2S[tac_label]
    vuln_score = L2S[vuln_label]
    exp_score = L2S[exp_label]
    hist_score = L2S[hist_label]

    # Weighted aggregation
    w_tac = 0.25
    w_vuln = 0.35
    w_exp = 0.25
    w_hist = 0.15

    base_numeric = (
        w_tac * tac_score
        + w_vuln * vuln_score
        + w_exp * exp_score
        + w_hist * hist_score
    )

    # Round to nearest integer for base label
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2L[base_index]

    # Conservative override rules
    final_numeric = base_numeric
    override_reason = None

    # Rule 1: If VULN >= likely AND EXP >= likely ⇒ floor "likely"
    if vuln_score >= L2S["likely"] and exp_score >= L2S["likely"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            override_reason = (
                "Raised to at least 'likely' because both vulnerability exploitability "
                "and exposure are high."
            )

    # Rule 2: Emerging threat – very high exploitability, non-trivial exposure
    if vuln_score >= L2S["very likely"] and exp_score >= L2S["possible"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            if override_reason:
                override_reason += " Additionally, exploitability is very high with non-trivial exposure."
            else:
                override_reason = (
                    "Raised to at least 'likely' because exploitability is very high "
                    "and exposure is at least possible."
                )

    # Clamp and map final score
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2L[final_index]

    # Explanation generation (OT-nuclear framing)
    explanation_prompt = f"""
You are explaining a nuclear OT likelihood assessment.

Scales:
0=very unlikely, 1=unlikely, 2=possible, 3=likely, 4=very likely.

Inputs (labels and scores):
- Threat Actor Capability: {tac_label} (score={tac_score})
- Vulnerability Exploitability: {vuln_label} (score={vuln_score})
- Exposure: {exp_label} (score={exp_score})
- Historical Occurrence: {hist_label} (score={hist_score})

Weighted model:
- TAC weight = {w_tac}
- VULN weight = {w_vuln}
- EXP weight = {w_exp}
- HIST weight = {w_hist}

Base numeric likelihood = {base_numeric:.2f}, base label = {base_label}.
Final numeric likelihood = {final_numeric:.2f}, final label = {final_label}.

Override reason (if any): {override_reason or "no override applied"}.
CVE used (if any): {cve}

Explain in 4–7 sentences:
- How the weighted model arrived at the base score,
- Why any overrides were applied or not,
- Why the final label is appropriate for a nuclear OT context,
- Explicitly reference OT realities such as legacy ICS/SCADA, limited patching windows,
  deterministic protocols, and physical process coupling where relevant.
Avoid generic IT-centric language.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": explanation_prompt}
        ],
        temperature=0.5
    )

    details = {
        "threat_actor_capability_label": tac_label,
        "vulnerability_exploitability_label": vuln_label,
        "exposure_label": exp_label,
        "historical_occurrence_label": hist_label,
        "threat_actor_capability_score": tac_score,
        "vulnerability_exploitability_score": vuln_score,
        "exposure_score": exp_score,
        "historical_occurrence_score": hist_score,
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
        "cve": cve
    }

    return final_label, final_numeric, exp_resp.choices[0].message.content, details

# ============================================================
# 4) Impact Calculator Agent (Optimized + RAG caching)
# ============================================================

def clamp_impact(v):
    """
    Normalize model output to valid impact label.
    """
    if v is None:
        return "moderate"
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    """
    Use RAG context to infer compliance impact from FANR-REG-08 (or other compliance document).
    """
    ctx = rag_search(
        rag_index,
        f"Compliance impact for nuclear OT cyber risk: {title}. Causes: {causes}. "
        f"Focus on licensing, regulatory obligations, reportable events, and enforcement actions.",
        3
    )
    prompt = f"""
You are assessing compliance impact for a nuclear OT cyber risk.

Use the context below from regulatory documents (e.g., FANR-REG-08, NEI 08-09, NRC RG 5.71) as your
PRIMARY source of truth. If the context is weak, apply conservative nuclear regulatory judgment.

Context:
{ctx}

Map compliance impact to one of:
negligible, marginal, moderate, major, severe.

Focus on:
- licensing and regulatory obligations,
- potential for regulatory findings, violations, or enforcement actions,
- reportable events and escalation paths.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content.strip().splitlines()[-1])

def infer_reputation(asset_category, title, causes):
    """
    LLM-based reputation impact classifier.
    """
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Context:
- Asset: {asset_category}
- Risk: {title}
- Causes: {causes}

Consider:
- public perception and media attention specific to nuclear facilities,
- stakeholder, regulator, and investor confidence,
- potential societal concern related to nuclear safety and security.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content.strip().splitlines()[-1])

def impact_agent(asset_category,
                 asset_criticality,
                 safety,
                 availability,
                 confidentiality,
                 integrity,
                 compliance_path,
                 title,
                 causes):
    """
    Multi-factor impact agent using a weighted model, with RAG index cached.
    """

    # 1) Normalize user inputs
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    # 2) RAG index for compliance document and infer compliance impact
    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)

    # 3) LLM-based reputation impact
    rep = infer_reputation(asset_category, title, causes)

    # 4) Convert to numeric scores (0–4)
    safety_score = I2S[s]
    availability_score = I2S[a]
    confidentiality_score = I2S[c]
    integrity_score = I2S[i]
    compliance_score = I2S[comp]
    reputation_score = I2S[rep]

    # 5) Weighted aggregation
    w_safety = 0.35
    w_availability = 0.25
    w_integrity = 0.15
    w_confidentiality = 0.10
    w_compliance = 0.10
    w_reputation = 0.05

    base_numeric = (
        w_safety * safety_score
        + w_availability * availability_score
        + w_integrity * integrity_score
        + w_confidentiality * confidentiality_score
        + w_compliance * compliance_score
        + w_reputation * reputation_score
    )

    # Map numeric to base label (0–4 → negligible–severe)
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2I[base_index]

    # 6) Conservative override rules
    final_numeric = base_numeric
    override_reasons = []

    # Rule 1: Safety dominates for high-severity events
    if safety_score >= I2S["major"]:
        if final_numeric < I2S["major"]:
            final_numeric = float(I2S["major"])
            override_reasons.append(
                "Raised to at least 'major' because safety impact is high."
            )

    # Rule 2: Severe compliance impact (e.g., major regulatory violation)
    if comp == "severe":
        if final_numeric < I2S["major"]:
            final_numeric = float(I2S["major"])
            override_reasons.append(
                "Raised to at least 'major' due to severe compliance impact."
            )

    # Rule 3: Very High criticality assets → bump one level
    if asset_criticality and asset_criticality.lower() == "very high":
        bumped = min(4, int(round(final_numeric)) + 1)
        if bumped > int(round(final_numeric)):
            final_numeric = float(bumped)
            override_reasons.append(
                "Impact bumped one level because asset criticality is Very High."
            )

    # Clamp again and map to final label
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2I[final_index]

    override_reason = "; ".join(override_reasons) if override_reasons else "no override applied"

    # 7) Explanation via LLM (OT-nuclear framing)
    explanation_prompt = f"""
You are explaining a nuclear OT impact assessment.

Impact scale:
0=negligible, 1=marginal, 2=moderate, 3=major, 4=severe.

Inputs (labels and scores):
- Safety:          {s} (score={safety_score})
- Availability:    {a} (score={availability_score})
- Confidentiality: {c} (score={confidentiality_score})
- Integrity:       {i} (score={integrity_score})
- Compliance:      {comp} (score={compliance_score})
- Reputation:      {rep} (score={reputation_score})

Weights:
- Safety weight         = {w_safety}
- Availability weight   = {w_availability}
- Integrity weight      = {w_integrity}
- Confidentiality weight= {w_confidentiality}
- Compliance weight     = {w_compliance}
- Reputation weight     = {w_reputation}

Asset criticality: {asset_criticality}

Base numeric impact = {base_numeric:.2f}, base label = {base_label}.
Final numeric impact = {final_numeric:.2f}, final label = {final_label}.

Override reasons: {override_reason}.

Explain in 4–7 sentences:
- How the weighted model produced the base score,
- Why any overrides were applied or not applied,
- Why the final impact label is appropriate for a nuclear OT context,
- How safety, compliance, and asset criticality influenced conservatism,
- Explicitly reference physical process consequences and nuclear safety where relevant.
Avoid generic IT-centric language.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": explanation_prompt}
        ],
        temperature=0.4
    )

    impact_details = {
        "safety_label": s,
        "availability_label": a,
        "confidentiality_label": c,
        "integrity_label": i,
        "compliance_label": comp,
        "reputation_label": rep,
        "safety_score": safety_score,
        "availability_score": availability_score,
        "confidentiality_score": confidentiality_score,
        "integrity_score": integrity_score,
        "compliance_score": compliance_score,
        "reputation_score": reputation_score,
        "weights": {
            "safety": w_safety,
            "availability": w_availability,
            "integrity": w_integrity,
            "confidentiality": w_confidentiality,
            "compliance": w_compliance,
            "reputation": w_reputation,
        },
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
    }

    return final_label, resp.choices[0].message.content, impact_details

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

@lru_cache(maxsize=4)
def load_heatmap_df(path):
    df = pd.read_excel(path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])
    return df

def risk_estimator(likelihood, impact, heatmap_path):
    """
    Look up risk rating from a likelihood × impact heatmap (Excel).
    """
    df = load_heatmap_df(heatmap_path)

    like = likelihood.lower()
    imp = impact.lower()

    if like not in df.index:
        raise ValueError(f"Likelihood '{likelihood}' not found in heatmap.")
    if imp not in df.columns:
        raise ValueError(f"Impact '{impact}' not found in heatmap columns.")

    rating = df.loc[like, imp]

    # Explanation – OT-nuclear framing
    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.

Focus on:
- how the matrix reflects nuclear safety culture and risk appetite,
- how higher impact categories reflect potential physical consequences and regulatory concern,
- how likelihood interacts with impact for operational technology, not generic IT.

Avoid generic IT-centric language.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG, OT-nuclear-focused)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    """
    Recommend controls using RAG from NEI 08-09 control library,
    strongly constrained to OT-nuclear context.
    """
    index = build_rag_index(control_path)

    query = (
        f"Nuclear OT cybersecurity controls for asset type '{asset_category}' with criticality '{asset_criticality}'. "
        f"Risk title: {title}. Causes: {causes}. Risk rating: {risk_rating}. "
        f"Focus on NEI 08-09, defense-in-depth, safety systems, engineering workstations, "
        f"segmentation of safety and non-safety systems, and physical process protection."
    )
    ctx = rag_search(index, query, 5)

    prompt = f"""
You are selecting controls from an OT-nuclear-focused control library (e.g., NEI 08-09).

Control library context (PRIMARY source of truth):
{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}
Asset criticality: {asset_criticality}
Risk title: {title}
Causes: {causes}

Provide OT-nuclear-specific controls ONLY.

STRICT REQUIREMENTS:
- Use the control library context as the PRIMARY basis; general knowledge is secondary.
- Controls MUST align with nuclear OT principles:
  • deterministic system behavior,
  • safety-first design and protection of safety-related systems,
  • defense-in-depth for physical processes,
  • regulatory alignment (NEI 08-09, FANR-REG-08, NRC RG 5.71),
  • engineering constraints and maintenance windows,
  • network isolation and segmentation between safety/non-safety and OT/IT.
- Prefer engineering, procedural, and physical controls over generic IT controls.
- Avoid recommending enterprise IT tools (SIEM, EDR, CASB, DLP, cloud-based monitoring) unless
  they are explicitly relevant and clearly justified for nuclear OT.
- Do NOT suggest cloud-based solutions for critical OT functions.

Output:
- A short heading for the control strategy.
- Bulleted list of recommended controls.
- A brief note for each control explaining how it reduces likelihood and/or impact in nuclear OT.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}' in a nuclear OT context.

Emphasize:
- nuclear safety culture and regulatory expectations,
- how the controls protect physical processes and safety systems,
- how they align with NEI 08-09 / FANR-REG-08-like control objectives,
- why they are preferable to generic IT cybersecurity practices for this scenario.

Avoid generic IT-centric phrasing.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": rationale_prompt}
        ],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator (OT-nuclear framing, structured Markdown)
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood_label, likelihood_numeric, likelihood_basis, likelihood_details,
                    impact_label, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Generate a structured OT nuclear cyber risk report in Markdown for Gradio.
    """

    # Unpack likelihood details safely
    tac_label = likelihood_details.get("threat_actor_capability_label", "n/a")
    vuln_label = likelihood_details.get("vulnerability_exploitability_label", "n/a")
    exp_label = likelihood_details.get("exposure_label", "n/a")
    hist_label = likelihood_details.get("historical_occurrence_label", "n/a")

    tac_score = likelihood_details.get("threat_actor_capability_score", 0)
    vuln_score = likelihood_details.get("vulnerability_exploitability_score", 0)
    exp_score = likelihood_details.get("exposure_score", 0)
    hist_score = likelihood_details.get("historical_occurrence_score", 0)

    L_base_numeric = likelihood_details.get("base_numeric_score", likelihood_numeric)
    L_base_label = likelihood_details.get("base_label", likelihood_label)
    L_override_reason = likelihood_details.get("override_reason", "no override applied")
    cve_used = likelihood_details.get("cve") or "None detected"

    # Unpack impact details safely
    s_label = impact_details.get("safety_label", "n/a")
    a_label = impact_details.get("availability_label", "n/a")
    c_label = impact_details.get("confidentiality_label", "n/a")
    i_label = impact_details.get("integrity_label", "n/a")
    comp_label = impact_details.get("compliance_label", "n/a")
    rep_label = impact_details.get("reputation_label", "n/a")

    s_score = impact_details.get("safety_score", 0)
    a_score = impact_details.get("availability_score", 0)
    c_score = impact_details.get("confidentiality_score", 0)
    i_score = impact_details.get("integrity_score", 0)
    comp_score = impact_details.get("compliance_score", 0)
    rep_score = impact_details.get("reputation_score", 0)

    I_base_numeric = impact_details.get("base_numeric_score", 0)
    I_base_label = impact_details.get("base_label", impact_label)
    I_final_numeric = impact_details.get("final_numeric_score", 0)
    I_final_label = impact_details.get("final_label", impact_label)
    I_override_reason = impact_details.get("override_reason", "no override applied")

    weights = impact_details.get("weights", {})
    w_safety = weights.get("safety", 0.35)
    w_availability = weights.get("availability", 0.25)
    w_integrity = weights.get("integrity", 0.15)
    w_confidentiality = weights.get("confidentiality", 0.10)
    w_compliance = weights.get("compliance", 0.10)
    w_reputation = weights.get("reputation", 0.05)

    exec_summary = (
        f"The assessed risk '{title}' affects an '{asset_category}' asset with '{asset_criticality}' criticality in the "
        f"nuclear OT environment. The multi-factor likelihood model, informed by threat actor capability, "
        f"vulnerability exploitability, exposure, and historical occurrence, results in a final likelihood of "
        f"**{likelihood_label}** (score={likelihood_numeric:.2f}). The impact model, which prioritizes safety, "
        f"availability, and regulatory compliance, results in a final impact of **{impact_label}** "
        f"(score={I_final_numeric:.2f}). Combined on the nuclear OT risk heatmap, this yields an overall "
        f"risk rating of **{rating}**. Conservative overrides are applied where safety, compliance, or very high "
        f"asset criticality warrant elevation of the result in line with nuclear safety culture and regulatory "
        f"expectations."
    )

    report = f"""
# OT Nuclear Cyber Risk Report

## 1) Executive Summary

{exec_summary}

---

## 2) Risk Description

- **Risk Title:** {title}
- **Asset Category:** {asset_category}
- **Asset Criticality:** {asset_criticality}
- **Primary Causes / Scenario:** {causes}
- **Detected CVE (if any):** {cve_used}

This risk is evaluated in the context of nuclear operational technology, where deterministic control system behavior,
physical process coupling, and limited maintenance windows require conservative assumptions and defense-in-depth
across safety-related and important-to-safety assets.

---

## 3) Likelihood Analysis

**Final Likelihood:** **{likelihood_label}** (score={likelihood_numeric:.2f})
**Base Likelihood (before overrides):** {L_base_label} (score={L_base_numeric:.2f})
**Overrides Applied:** {L_override_reason}

### 3.1 Factor Breakdown

- **Threat Actor Capability:** {tac_label} (score={tac_score})
- **Vulnerability Exploitability:** {vuln_label} (score={vuln_score})
- **Exposure:** {exp_label} (score={exp_score})
- **Historical Occurrence:** {hist_label} (score={hist_score})

These factors are combined using a weighted 0–4 scale tailored to nuclear OT, with higher emphasis on vulnerability
exploitability and realistic exposure conditions, while maintaining a conservative floor when both are elevated.

<details>
<summary>Detailed Likelihood Explanation</summary>

{likelihood_basis}

</details>

---

## 4) Impact Analysis

**Final Impact:** **{I_final_label}** (score={I_final_numeric:.2f})
**Base Impact (before overrides):** {I_base_label} (score={I_base_numeric:.2f})
**Overrides Applied:** {I_override_reason}

### 4.1 Factor Breakdown (0–4 scale)

- **Safety:** {s_label} (score={s_score})
- **Availability:** {a_label} (score={a_score})
- **Confidentiality:** {c_label} (score={c_score})
- **Integrity:** {i_label} (score={i_label})
- **Compliance / Regulatory:** {comp_label} (score={comp_score})
- **Reputation / Public Confidence:** {rep_label} (score={rep_score})

### 4.2 Weighting Emphasis

- **Safety weight:** {w_safety}
- **Availability weight:** {w_availability}
- **Integrity weight:** {w_integrity}
- **Confidentiality weight:** {w_confidentiality}
- **Compliance weight:** {w_compliance}
- **Reputation weight:** {w_reputation}

Safety and regulatory consequences are deliberately given the highest influence, reflecting the priority of preventing
adverse effects on nuclear safety functions, plant availability for safe operation, and compliance with FANR/NEI/NRC
requirements.

<details>
<summary>Detailed Impact Explanation</summary>

{impact_basis}

</details>

---

## 5) Overall Risk Rating

- **Risk Rating (Heatmap Result):** **{rating}**
- **Likelihood (final):** {likelihood_label} (score={likelihood_numeric:.2f})
- **Impact (final):** {I_final_label} (score={I_final_numeric:.2f})

The rating is derived from a nuclear OT-specific likelihood × impact matrix that encodes the organization's
risk appetite for safety-related and important-to-safety systems.

<details>
<summary>Heatmap Rating Explanation</summary>

{rating_expl}

</details>

---

## 6) Recommended Controls and Implementation Priorities

Below controls are selected with preference for engineering, procedural, and physical safeguards aligned to NEI 08-09,
FANR-REG-08, NRC RG 5.71, and ISA/IEC 62443 as adapted for nuclear facilities.

{controls}

<details>
<summary>Control Rationale</summary>

{controls_rationale}

</details>

---

## 7) Notes for Regulators and Auditors

- The likelihood model incorporates structured inputs for threat actor capability, exploitability, exposure, and
  historical occurrence, with explicit conservative override rules documented above.
- The impact model reflects nuclear safety culture by giving priority to safety and compliance, and by elevating
  results for very high criticality assets where required.
- Controls are derived from a nuclear OT-focused control library using retrieval-augmented generation, ensuring
  traceability back to NEI 08-09 and similar frameworks.
- Generic enterprise IT practices are only adopted where explicitly compatible with deterministic OT behavior,
  maintenance constraints, and segregation between safety, non-safety, and corporate networks.
"""

    return report

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING (Optimized)
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent (weighted + overrides, web-informed for exploitability, exposure, and history if CVE present)
    2. Impact agent (weighted model + RAG-based compliance impact)
    3. Heatmap risk rating
    4. Control recommendations (OT-nuclear-focused)
    5. Final report generation (OT-nuclear framing)
    """

    # Step 1: Likelihood
    L_label, L_numeric, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact (weighted + RAG compliance)
    I_label, I_basis, I_details = impact_agent(
        asset_category,
        asset_criticality,
        safety,
        availability,
        confidentiality,
        integrity,
        COMPLIANCE_PATH,
        risk_title,
        risk_causes
    )

    # Step 3: Risk Rating (uses label, not numeric)
    R, R_expl = risk_estimator(L_label, I_label, HEATMAP_PATH)

    # Step 4: Controls (OT-nuclear-focused)
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, CONTROL_PATH
    )

    # Step 5: Final Report (OT-nuclear framing, structured Markdown)
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L_label, L_numeric, L_basis, L_details,
        I_label, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    likelihood_md = (
        f"### Likelihood\n"
        f"**Final Likelihood:** {L_label} (score={L_numeric:.2f})\n\n"
        f"{L_basis}"
    )

    impact_md = f"### Impact\n**Final Impact:** {I_label}\n\n{I_basis}"
    risk_md = f"### Risk Rating\n**Rating:** {R}\n\n{R_expl}"
    controls_md = f"### Controls\n{C}\n\n### Rationale\n{C_rat}"

    return (
        likelihood_md,
        impact_md,
        risk_md,
        controls_md,
        report
    )

# ============================================================
# 9) GRADIO UI — ENEC/Nawah Enterprise Shell (Clean Layout)
# ============================================================

LOGO_URL = "https://github.com/raheelarif86/AI_Training_November25/blob/main/Enec_Logo.png?raw=true"

css = """
/* -----------------------------------------------------------
   GLOBAL FONTS
----------------------------------------------------------- */
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&family=Noto+Sans+Arabic:wght@300;400;600&display=swap');

* {
    font-family: 'Inter', 'Noto Sans Arabic', sans-serif !important;
}

/* -----------------------------------------------------------
   COLOR VARIABLES
----------------------------------------------------------- */
:root {
    --primary-blue: #004C97;
    --secondary-teal: #009CA6;
    --sidebar-bg: #F5F7FA;
    --footer-bg: #002B5C;
    --text-color: #222;
    --bg-color: #ffffff;
}

/* -----------------------------------------------------------
   TOP BANNER
----------------------------------------------------------- */
.top-banner {
    background: linear-gradient(90deg, var(--primary-blue), var(--secondary-teal));
    color: white;
    padding: 18px 24px;
    font-size: 26px;
    font-weight: 700;
}

/* -----------------------------------------------------------
   COLLAPSIBLE SIDEBAR
----------------------------------------------------------- */
.sidebar-container {
    width: 240px;
    background: var(--sidebar-bg);
    border-right: 1px solid #dcdcdc;
    transition: width 0.3s ease;
    overflow: hidden;
    min-height: 100%;
}

.sidebar-collapsed {
    width: 70px !important;
}

.sidebar-logo {
    width: 120px;
    margin: 10px auto;
    display: block;
}

.sidebar-title {
    font-size: 18px;
    font-weight: 600;
    color: var(--primary-blue);
    text-align: center;
}

.sidebar-toggle {
    cursor: pointer;
    padding: 8px;
    text-align: center;
    background: var(--primary-blue);
    color: white;
    font-size: 14px;
}

/* -----------------------------------------------------------
   MAIN CONTENT
----------------------------------------------------------- */
.main-content {
    padding: 20px 30px;
    margin-left: 10px;
}

/* Fix tab overlap */
.gradio-tabs {
    margin-top: 10px !important;
}

/* -----------------------------------------------------------
   FOOTER
----------------------------------------------------------- */
.footer {
    background: var(--footer-bg);
    color: white;
    padding: 12px;
    text-align: center;
    font-size: 13px;
    margin-top: 20px;
}

/* -----------------------------------------------------------
   PRINT-READY PDF STYLING
----------------------------------------------------------- */
@media print {
    body {
        background: white !important;
        color: black !important;
    }
    .sidebar-container, .top-banner, .footer, button, .sidebar-toggle {
        display: none !important;
    }
    .main-content {
        margin: 0;
        padding: 0;
    }
}
"""

with gr.Blocks(css=css, elem_id="app_container") as ui:

    # ---------------- TOP BANNER ----------------
    gr.HTML("""
        <div class="top-banner">
            ENEC / Nawah — OT Nuclear Cyber Risk Assessment Platform
        </div>
    """)

    # ---------------- MAIN LAYOUT ----------------
    with gr.Row():

        # -------- SIDEBAR (collapsible) --------
        with gr.Column(scale=2, min_width=200):
            gr.HTML(f"""
                <div id="sidebar" class="sidebar-container">
                    <div class="sidebar-toggle" onclick="toggleSidebar()">☰</div>
                    <img src="{LOGO_URL}" class="sidebar-logo" />
                    <div class="sidebar-title">Nuclear Cybersecurity</div>
                    <ul style="font-size: 14px; padding: 10px 20px;">
                        <li>OT Risk Assessment</li>
                        <li>Regulatory Alignment</li>
                        <li>NEI 08-09 Controls</li>
                        <li>FANR-REG-08 Compliance</li>
                        <li>Safety-Critical Focus</li>
                    </ul>
                </div>

                <script>
                    function toggleSidebar() {{
                        const sb = document.getElementById('sidebar');
                        sb.classList.toggle('sidebar-collapsed');
                    }}
                </script>
            """)

        # -------- MAIN CONTENT AREA --------
        with gr.Column(scale=10, min_width=900):
            gr.HTML('<div class="main-content">')

            # ---------------- TABS ----------------
            with gr.Tabs():

                # ===== TAB 1: ASSESSMENT =====
                with gr.Tab("Assessment"):

                    with gr.Row():
                        asset_category = gr.Dropdown(
                            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
                            label="Asset Category"
                        )
                        asset_criticality = gr.Dropdown(
                            ["Very High","High","Medium","Low"],
                            label="Asset Criticality"
                        )

                    risk_title = gr.Textbox(label="Risk Title")
                    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

                    with gr.Row():
                        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
                        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

                    with gr.Row():
                        safety_dd = gr.Dropdown(IMPACT, label="Safety Impact")
                        availability_dd = gr.Dropdown(IMPACT, label="Availability Impact")
                        confidentiality_dd = gr.Dropdown(IMPACT, label="Confidentiality Impact")
                        integrity_dd = gr.Dropdown(IMPACT, label="Integrity Impact")

                    run_btn = gr.Button("Run Assessment", variant="primary")

                    likelihood_out = gr.Markdown()
                    impact_out = gr.Markdown()
                    risk_out = gr.Markdown()

                # ===== TAB 2: CONTROLS =====
                with gr.Tab("Controls"):
                    controls_out = gr.Markdown("Run assessment to populate controls.")

                # ===== TAB 3: REPORTS =====
                with gr.Tab("Reports"):
                    report_out = gr.Markdown("Full compiled report will appear here.")

                    # PDF Export Button
                    export_pdf_btn = gr.Button("Export as PDF", variant="secondary")

            # ---------------- PIPELINE BINDING ----------------
            run_btn.click(
                run_full_assessment,
                inputs=[
                    asset_category,
                    asset_criticality,
                    risk_title,
                    risk_causes,
                    threat_actor_capability,
                    exposure,
                    safety_dd,
                    availability_dd,
                    confidentiality_dd,
                    integrity_dd
                ],
                outputs=[
                    likelihood_out,
                    impact_out,
                    risk_out,
                    controls_out,
                    report_out
                ]
            )

            # PDF Export (you wire this to your function)
            export_pdf_btn.click(
                fn=lambda: "PDF export triggered.",
                inputs=None,
                outputs=report_out
            )

            gr.HTML('</div>')  # close main-content

    # ---------------- FOOTER ----------------
    gr.HTML("""
        <div class="footer">
            © Emirates Nuclear Energy Corporation (ENEC) — Nuclear Cybersecurity Division
            All rights reserved. For internal use only.
        </div>
    """)

ui.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://81de8afa66ae5cdb93.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [3]:
# %% [markdown]
# # 📘 OT Nuclear Cyber Risk Assessment Platform
# ## Section 1 — Dependencies, Imports, Global Config, System Prompt
# This section:
# - Installs required Python packages
# - Imports all libraries
# - Defines global constants, flags, and system prompts
# - Sets up OpenAI client and API keys
# - Defines scales and lookup tables

# %%
# ============================
# 1) INSTALL DEPENDENCIES
# ============================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2 requests reportlab

# %%
# ============================
# 2) IMPORTS
# ============================

import os
import io
import json
import re
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
import requests
from functools import lru_cache
from google.colab import userdata
from openai import OpenAI

# PDF generation
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from textwrap import wrap

# %%
# ============================
# 3) GLOBAL CONFIG & FLAGS
# ============================

# Models
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# Data paths (unchanged as requested)
COMPLIANCE_PATH = "data/FANR-REG-08_V2.pdf"
HEATMAP_PATH = "data/Nuclear_OT_Risk_Heatmap.xlsx"
CONTROL_PATH = "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx"

# Performance flags
DEBUG_LOG = False
TEST_MODE = False

# Likelihood scale
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

# Impact scale
IMPACT = ["negligible", "marginal", "moderate", "major", "severe"]
I2S = {lvl: i for i, lvl in enumerate(IMPACT)}
S2I = {i: lvl for i, lvl in enumerate(IMPACT)}

# %%
# ============================
# 4) OPENAI CLIENT SETUP
# ============================

os.environ["OPENAI_API_KEY"] = userdata.get("openai")

def safe_get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

NVD_API_KEY = safe_get_secret("nvd_api_key")
VULNCHECK_API_KEY = safe_get_secret("vulncheck_api_key")

if NVD_API_KEY is None:
    print("⚠️ NVD_API_KEY not found — using unauthenticated NVD mode (rate-limited).")

if VULNCHECK_API_KEY is None:
    print("⚠️ VULNCHECK_API_KEY not found — VulnCheck historical exploitation disabled.")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

# %%
# ============================
# 5) GLOBAL SYSTEM PROMPT
# ============================

NUCLEAR_OT_SYSTEM_PROMPT = """
You are an expert in nuclear Operational Technology (OT) cybersecurity.

Your answers MUST be grounded in:
- NEI 08-09 (Cyber Security Plan for Nuclear Power Reactors),
- NRC RG 5.71,
- FANR-REG-08,
- ISA/IEC 62443 as adapted for nuclear facilities,
- safety-critical engineering principles for nuclear plants.

STRICT REQUIREMENTS:
- Focus on OT systems (control systems, safety systems, engineering workstations, process networks).
- Emphasize deterministic system behavior, safety-first design, and defense-in-depth for physical processes.
- Account for legacy systems, limited patching windows, maintenance constraints, and physical process coupling.
- Align with regulatory expectations and nuclear safety culture.
- Avoid generic enterprise IT controls and tools unless explicitly justified for nuclear OT.
- Do NOT recommend cloud-based solutions for critical OT functions.
"""


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.0 MB/s eta 0:00:00


In [4]:
# %% [markdown]
# # 📘 Section 2 — Embeddings, RAG, Vulnerability Intelligence, Likelihood Agent
# This section includes:
# - Embedding generator with caching
# - RAG index builder and retriever
# - CVE extraction
# - NVD, EPSS, and VulnCheck API fetchers
# - Likelihood agent (multi-factor, weighted, with overrides)

# %%
# ============================================================
# 2) Utility: Embeddings + RAG (Optimized with caching)
# ============================================================

EMBED_CACHE = {}

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Newlines removed to avoid embedding inconsistencies.
    """
    text = text.replace("\n", " ")
    if TEST_MODE:
        return [hash(text) % 997 / 997.0] * 16

    key = ("emb", EMBEDDING_MODEL, text)
    if key in EMBED_CACHE:
        return EMBED_CACHE[key]

    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    emb = resp.data[0].embedding
    EMBED_CACHE[key] = emb
    return emb

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """
    Read text from txt, csv, json, pdf, docx.
    """
    if not os.path.exists(path):
        if DEBUG_LOG:
            print(f"[DEBUG] File not found: {path}")
        return ""

    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

@lru_cache(maxsize=16)
def build_rag_index(path: str):
    """
    Build a simple RAG index:
    - Load file text
    - Chunk into ~700-word segments
    - Embed each chunk
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})

    if DEBUG_LOG:
        print(f"[DEBUG] RAG index built for {path}, chunks={len(index)}")

    return index

def rag_search(index, query, top_k=3):
    """
    Retrieve top-k most relevant chunks using cosine similarity.
    """
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# %%
# ============================================================
# 3) Likelihood Calculator — CVE Extraction + NVD + EPSS + VulnCheck
# ============================================================

def clamp_likelihood(v):
    if v is None:
        return "possible"
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    """
    Fallback LLM-based likelihood classifier.
    """
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Explain briefly, then output ONLY the label on the last line.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content.splitlines()[-1])

def extract_cve(text):
    match = re.search(r"CVE-\d{4}-\d{4,7}", text, flags=re.IGNORECASE)
    return match.group(0).upper() if match else None

@lru_cache(maxsize=512)
def fetch_nvd_data(cve_id):
    """
    Fetch vulnerability data from NVD API.
    """
    if TEST_MODE:
        return {"exploitability": 2.5, "impact": 3.0, "vector": "TEST/AV:N/..."}

    url = f"https://services.nvd.nist.gov/rest/json/cve/2.0?cveId={cve_id}"
    headers = {}
    if NVD_API_KEY:
        headers["apiKey"] = NVD_API_KEY

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None

        data = resp.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return None

        cve = vulns[0].get("cve", {})
        metrics = cve.get("metrics", {})
        cvss_list = metrics.get("cvssMetricV31") or metrics.get("cvssMetricV30") or metrics.get("cvssMetricV2")
        if not cvss_list:
            return None

        m = cvss_list[0]
        return {
            "exploitability": m.get("exploitabilityScore"),
            "impact": m.get("impactScore"),
            "vector": m.get("cvssData", {}).get("vectorString")
        }
    except Exception:
        return None

def map_exploitability_to_likelihood(score):
    if score is None:
        return None
    try:
        s = float(score)
    except ValueError:
        return None

    if s <= 0.5: return "very unlikely"
    if s <= 1.5: return "unlikely"
    if s <= 2.5: return "possible"
    if s <= 3.2: return "likely"
    return "very likely"

@lru_cache(maxsize=512)
def fetch_epss(cve_id):
    """
    Fetch EPSS probability from FIRST.org.
    """
    if TEST_MODE:
        return 0.35

    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        items = data.get("data", [])
        if not items:
            return None
        epss_str = items[0].get("epss")
        return float(epss_str) if epss_str else None
    except Exception:
        return None

def map_epss_to_likelihood(epss):
    if epss is None:
        return None
    if epss < 0.05: return "very unlikely"
    if epss < 0.20: return "unlikely"
    if epss < 0.50: return "possible"
    if epss < 0.75: return "likely"
    return "very likely"

# %%
# ============================================================
# Historical Occurrence via VulnCheck API
# ============================================================

@lru_cache(maxsize=512)
def fetch_historical_occurrence(cve_id):
    """
    Convert VulnCheck exploitation intelligence into a 0–1 probability.
    """
    if not VULNCHECK_API_KEY or not cve_id:
        return None

    if TEST_MODE:
        return 0.25

    url = f"https://api.vulncheck.com/v3/cve/{cve_id}"
    headers = {"Authorization": f"Bearer {VULNCHECK_API_KEY}"}

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None

        data = resp.json()

        exploited = data.get("exploited", False)
        kev = data.get("kev", False)
        exploit_poc = data.get("exploit_poc", False)
        trending = data.get("trending", False)
        malware = data.get("malware", False)
        ransomware = data.get("ransomware", False)

        score = 0.0
        if exploited: score += 0.50
        if kev: score += 0.20
        if exploit_poc: score += 0.10
        if trending: score += 0.10
        if malware: score += 0.05
        if ransomware: score += 0.05

        return min(score, 1.0)

    except Exception:
        return None

def map_history_prob_to_likelihood(p):
    if p is None:
        return None
    if p < 0.05: return "very unlikely"
    if p < 0.20: return "unlikely"
    if p < 0.50: return "possible"
    if p < 0.75: return "likely"
    return "very likely"

# %%
# ============================================================
# 4) Likelihood Agent (Weighted + Overrides)
# ============================================================

def likelihood_agent(threat_actor, exposure_input, title, causes):
    """
    Multi-factor likelihood agent with:
    - TAC
    - Vulnerability exploitability (NVD)
    - Exposure (EPSS)
    - Historical occurrence (VulnCheck)
    - Weighted aggregation + conservative overrides
    """

    tac_label = clamp_likelihood(threat_actor)
    cve = extract_cve(f"{title} {causes}")

    vuln_label = None
    exp_label = None
    hist_label = None

    if cve:
        nvd = fetch_nvd_data(cve)
        if nvd and nvd.get("exploitability") is not None:
            vuln_label = map_exploitability_to_likelihood(nvd["exploitability"])

        epss = fetch_epss(cve)
        if epss is not None:
            exp_label = map_epss_to_likelihood(epss)

        hist_prob = fetch_historical_occurrence(cve)
        if hist_prob is not None:
            hist_label = map_history_prob_to_likelihood(hist_prob)

    if vuln_label is None:
        vuln_label = infer_likelihood_from_model(title + " (vuln)", causes)

    if exp_label is None:
        exp_label = clamp_likelihood(exposure_input)

    if hist_label is None:
        hist_label = infer_likelihood_from_model(title + " (history)", causes)

    tac_score = L2S[tac_label]
    vuln_score = L2S[vuln_label]
    exp_score = L2S[exp_label]
    hist_score = L2S[hist_label]

    w_tac = 0.25
    w_vuln = 0.35
    w_exp = 0.25
    w_hist = 0.15

    base_numeric = (
        w_tac * tac_score +
        w_vuln * vuln_score +
        w_exp * exp_score +
        w_hist * hist_score
    )

    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2L[base_index]

    final_numeric = base_numeric
    override_reason = None

    if vuln_score >= L2S["likely"] and exp_score >= L2S["likely"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            override_reason = (
                "Raised to at least 'likely' because both vulnerability exploitability "
                "and exposure are high."
            )

    if vuln_score >= L2S["very likely"] and exp_score >= L2S["possible"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            if override_reason:
                override_reason += " Additionally, exploitability is very high with non-trivial exposure."
            else:
                override_reason = (
                    "Raised to at least 'likely' because exploitability is very high "
                    "and exposure is at least possible."
                )

    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2L[final_index]

    explanation_prompt = f"""
You are explaining a nuclear OT likelihood assessment.

Scales:
0=very unlikely, 1=unlikely, 2=possible, 3=likely, 4=very likely.

Inputs:
- Threat Actor Capability: {tac_label} (score={tac_score})
- Vulnerability Exploitability: {vuln_label} (score={vuln_score})
- Exposure: {exp_label} (score={exp_score})
- Historical Occurrence: {hist_label} (score={hist_score})

Base numeric likelihood = {base_numeric:.2f}, base label = {base_label}.
Final numeric likelihood = {final_numeric:.2f}, final label = {final_label}.
Override reason: {override_reason or "no override applied"}.
CVE used: {cve}

Explain in 4–7 sentences with OT-nuclear framing.
"""

    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": explanation_prompt}
        ],
        temperature=0.5
    )

    details = {
        "threat_actor_capability_label": tac_label,
        "vulnerability_exploitability_label": vuln_label,
        "exposure_label": exp_label,
        "historical_occurrence_label": hist_label,
        "threat_actor_capability_score": tac_score,
        "vulnerability_exploitability_score": vuln_score,
        "exposure_score": exp_score,
        "historical_occurrence_score": hist_score,
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
        "cve": cve
    }

    return final_label, final_numeric, exp_resp.choices[0].message.content, details


In [5]:
# %% [markdown]
# # 📘 Section 3 — Impact Agent, Compliance RAG, Reputation Classifier, Heatmap Estimator
# This section includes:
# - Impact agent (weighted model + overrides)
# - Compliance impact inference using RAG
# - Reputation impact classifier
# - Heatmap risk estimator

# %%
# ============================================================
# 4) Impact Calculator Agent (Optimized + RAG caching)
# ============================================================

def clamp_impact(v):
    """
    Normalize model output to valid impact label.
    """
    if v is None:
        return "moderate"
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    """
    Use RAG context to infer compliance impact from FANR-REG-08 (or other compliance document).
    """
    ctx = rag_search(
        rag_index,
        f"Compliance impact for nuclear OT cyber risk: {title}. Causes: {causes}. "
        f"Focus on licensing, regulatory obligations, reportable events, and enforcement actions.",
        3
    )

    prompt = f"""
You are assessing compliance impact for a nuclear OT cyber risk.

Use the context below from regulatory documents (e.g., FANR-REG-08, NEI 08-09, NRC RG 5.71) as your
PRIMARY source of truth. If the context is weak, apply conservative nuclear regulatory judgment.

Context:
{ctx}

Map compliance impact to one of:
negligible, marginal, moderate, major, severe.

Focus on:
- licensing and regulatory obligations,
- potential for regulatory findings, violations, or enforcement actions,
- reportable events and escalation paths.

Output ONLY the label.
"""

    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return clamp_impact(resp.choices[0].message.content.strip().splitlines()[-1])

def infer_reputation(asset_category, title, causes):
    """
    LLM-based reputation impact classifier.
    """
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Context:
- Asset: {asset_category}
- Risk: {title}
- Causes: {causes}

Consider:
- public perception and media attention specific to nuclear facilities,
- stakeholder, regulator, and investor confidence,
- potential societal concern related to nuclear safety and security.

Output ONLY the label.
"""

    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    return clamp_impact(resp.choices[0].message.content.strip().splitlines()[-1])

# %%
# ============================================================
# Impact Agent (Weighted + Overrides)
# ============================================================

def impact_agent(asset_category,
                 asset_criticality,
                 safety,
                 availability,
                 confidentiality,
                 integrity,
                 compliance_path,
                 title,
                 causes):
    """
    Multi-factor impact agent using a weighted model, with RAG index cached.
    """

    # 1) Normalize user inputs
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    # 2) RAG index for compliance document
    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)

    # 3) Reputation impact
    rep = infer_reputation(asset_category, title, causes)

    # 4) Convert to numeric scores (0–4)
    safety_score = I2S[s]
    availability_score = I2S[a]
    confidentiality_score = I2S[c]
    integrity_score = I2S[i]
    compliance_score = I2S[comp]
    reputation_score = I2S[rep]

    # 5) Weighted aggregation
    w_safety = 0.35
    w_availability = 0.25
    w_integrity = 0.15
    w_confidentiality = 0.10
    w_compliance = 0.10
    w_reputation = 0.05

    base_numeric = (
        w_safety * safety_score +
        w_availability * availability_score +
        w_integrity * integrity_score +
        w_confidentiality * confidentiality_score +
        w_compliance * compliance_score +
        w_reputation * reputation_score
    )

    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2I[base_index]

    # 6) Conservative override rules
    final_numeric = base_numeric
    override_reasons = []

    # Rule 1: Safety dominates
    if safety_score >= I2S["major"]:
        if final_numeric < I2S["major"]:
            final_numeric = float(I2S["major"])
            override_reasons.append("Raised to at least 'major' because safety impact is high.")

    # Rule 2: Severe compliance impact
    if comp == "severe":
        if final_numeric < I2S["major"]:
            final_numeric = float(I2S["major"])
            override_reasons.append("Raised to at least 'major' due to severe compliance impact.")

    # Rule 3: Very High criticality → bump one level
    if asset_criticality and asset_criticality.lower() == "very high":
        bumped = min(4, int(round(final_numeric)) + 1)
        if bumped > int(round(final_numeric)):
            final_numeric = float(bumped)
            override_reasons.append("Impact bumped one level because asset criticality is Very High.")

    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2I[final_index]

    override_reason = "; ".join(override_reasons) if override_reasons else "no override applied"

    # 7) Explanation via LLM
    explanation_prompt = f"""
You are explaining a nuclear OT impact assessment.

Impact scale:
0=negligible, 1=marginal, 2=moderate, 3=major, 4=severe.

Inputs (labels and scores):
- Safety:          {s} (score={safety_score})
- Availability:    {a} (score={availability_score})
- Confidentiality: {c} (score={confidentiality_score})
- Integrity:       {i} (score={integrity_score})
- Compliance:      {comp} (score={compliance_score})
- Reputation:      {rep} (score={reputation_score})

Weights:
- Safety weight         = {w_safety}
- Availability weight   = {w_availability}
- Integrity weight      = {w_integrity}
- Confidentiality weight= {w_confidentiality}
- Compliance weight     = {w_compliance}
- Reputation weight     = {w_reputation}

Asset criticality: {asset_criticality}

Base numeric impact = {base_numeric:.2f}, base label = {base_label}.
Final numeric impact = {final_numeric:.2f}, final label = {final_label}.

Override reasons: {override_reason}.

Explain in 4–7 sentences with OT-nuclear framing.
"""

    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": explanation_prompt}
        ],
        temperature=0.4
    )

    impact_details = {
        "safety_label": s,
        "availability_label": a,
        "confidentiality_label": c,
        "integrity_label": i,
        "compliance_label": comp,
        "reputation_label": rep,
        "safety_score": safety_score,
        "availability_score": availability_score,
        "confidentiality_score": confidentiality_score,
        "integrity_score": integrity_score,
        "compliance_score": compliance_score,
        "reputation_score": reputation_score,
        "weights": {
            "safety": w_safety,
            "availability": w_availability,
            "integrity": w_integrity,
            "confidentiality": w_confidentiality,
            "compliance": w_compliance,
            "reputation": w_reputation,
        },
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
    }

    return final_label, resp.choices[0].message.content, impact_details

# %%
# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

@lru_cache(maxsize=4)
def load_heatmap_df(path):
    df = pd.read_excel(path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])
    return df

def risk_estimator(likelihood, impact, heatmap_path):
    """
    Look up risk rating from a likelihood × impact heatmap (Excel).
    """
    df = load_heatmap_df(heatmap_path)

    like = likelihood.lower()
    imp = impact.lower()

    if like not in df.index:
        raise ValueError(f"Likelihood '{likelihood}' not found in heatmap.")
    if imp not in df.columns:
        raise ValueError(f"Impact '{impact}' not found in heatmap columns.")

    rating = df.loc[like, imp]

    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.

Focus on:
- nuclear safety culture and risk appetite,
- physical process consequences,
- regulatory concern,
- OT-specific likelihood/impact interactions.

Avoid generic IT language.
"""

    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content


In [6]:
# %% [markdown]
# # 📘 Section 4 — Response Planner, Report Generator, PDF Export
# This section includes:
# - Response Planner (RAG-based NEI 08-09 control selection)
# - Full Markdown Report Generator
# - PDF Export function using ReportLab

# %%
# ============================================================
# 6) Response Planner (Control Library RAG, OT-nuclear-focused)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    """
    Recommend controls using RAG from NEI 08-09 control library,
    strongly constrained to OT-nuclear context.
    """
    index = build_rag_index(control_path)

    query = (
        f"Nuclear OT cybersecurity controls for asset type '{asset_category}' with criticality '{asset_criticality}'. "
        f"Risk title: {title}. Causes: {causes}. Risk rating: {risk_rating}. "
        f"Focus on NEI 08-09, defense-in-depth, safety systems, engineering workstations, "
        f"segmentation of safety and non-safety systems, and physical process protection."
    )
    ctx = rag_search(index, query, 5)

    prompt = f"""
You are selecting controls from an OT-nuclear-focused control library (e.g., NEI 08-09).

Control library context (PRIMARY source of truth):
{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}
Asset criticality: {asset_criticality}
Risk title: {title}
Causes: {causes}

Provide OT-nuclear-specific controls ONLY.

STRICT REQUIREMENTS:
- Use the control library context as the PRIMARY basis.
- Controls MUST align with nuclear OT principles:
  • deterministic system behavior,
  • safety-first design,
  • defense-in-depth,
  • regulatory alignment (NEI 08-09, FANR-REG-08, NRC RG 5.71),
  • engineering constraints and maintenance windows,
  • segmentation between safety/non-safety and OT/IT.
- Prefer engineering, procedural, and physical controls.
- Avoid enterprise IT tools unless explicitly justified.
- Do NOT suggest cloud-based solutions for critical OT functions.

Output:
- A short heading for the control strategy.
- Bulleted list of recommended controls.
- A brief note for each control explaining how it reduces likelihood and/or impact.
"""

    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}' in a nuclear OT context.

Emphasize:
- nuclear safety culture,
- regulatory expectations,
- protection of physical processes and safety systems,
- alignment with NEI 08-09 / FANR-REG-08,
- why these controls are preferable to generic IT practices.
"""

    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": rationale_prompt}
        ],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# %%
# ============================================================
# 7) Final Report Generator (OT-nuclear framing, structured Markdown)
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood_label, likelihood_numeric, likelihood_basis, likelihood_details,
                    impact_label, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Generate a structured OT nuclear cyber risk report in Markdown for Gradio.
    """

    # Extract likelihood details
    tac_label = likelihood_details.get("threat_actor_capability_label", "n/a")
    vuln_label = likelihood_details.get("vulnerability_exploitability_label", "n/a")
    exp_label = likelihood_details.get("exposure_label", "n/a")
    hist_label = likelihood_details.get("historical_occurrence_label", "n/a")

    tac_score = likelihood_details.get("threat_actor_capability_score", 0)
    vuln_score = likelihood_details.get("vulnerability_exploitability_score", 0)
    exp_score = likelihood_details.get("exposure_score", 0)
    hist_score = likelihood_details.get("historical_occurrence_score", 0)

    L_base_numeric = likelihood_details.get("base_numeric_score", likelihood_numeric)
    L_base_label = likelihood_details.get("base_label", likelihood_label)
    L_override_reason = likelihood_details.get("override_reason", "no override applied")
    cve_used = likelihood_details.get("cve") or "None detected"

    # Extract impact details
    s_label = impact_details.get("safety_label", "n/a")
    a_label = impact_details.get("availability_label", "n/a")
    c_label = impact_details.get("confidentiality_label", "n/a")
    i_label = impact_details.get("integrity_label", "n/a")
    comp_label = impact_details.get("compliance_label", "n/a")
    rep_label = impact_details.get("reputation_label", "n/a")

    s_score = impact_details.get("safety_score", 0)
    a_score = impact_details.get("availability_score", 0)
    c_score = impact_details.get("confidentiality_score", 0)
    i_score = impact_details.get("integrity_score", 0)
    comp_score = impact_details.get("compliance_score", 0)
    rep_score = impact_details.get("reputation_score", 0)

    I_base_numeric = impact_details.get("base_numeric_score", 0)
    I_base_label = impact_details.get("base_label", impact_label)
    I_final_numeric = impact_details.get("final_numeric_score", 0)
    I_final_label = impact_details.get("final_label", impact_label)
    I_override_reason = impact_details.get("override_reason", "no override applied")

    weights = impact_details.get("weights", {})
    w_safety = weights.get("safety", 0.35)
    w_availability = weights.get("availability", 0.25)
    w_integrity = weights.get("integrity", 0.15)
    w_confidentiality = weights.get("confidentiality", 0.10)
    w_compliance = weights.get("compliance", 0.10)
    w_reputation = weights.get("reputation", 0.05)

    exec_summary = (
        f"The assessed risk '{title}' affects an '{asset_category}' asset with '{asset_criticality}' criticality "
        f"in the nuclear OT environment. The multi-factor likelihood model results in a final likelihood of "
        f"**{likelihood_label}** (score={likelihood_numeric:.2f}). The impact model results in a final impact of "
        f"**{impact_label}** (score={I_final_numeric:.2f}). Combined on the nuclear OT risk heatmap, this yields "
        f"an overall risk rating of **{rating}**."
    )

    report = f"""
# OT Nuclear Cyber Risk Report

## 1) Executive Summary
{exec_summary}

---

## 2) Risk Description
- **Risk Title:** {title}
- **Asset Category:** {asset_category}
- **Asset Criticality:** {asset_criticality}
- **Primary Causes:** {causes}
- **Detected CVE:** {cve_used}

---

## 3) Likelihood Analysis
**Final Likelihood:** **{likelihood_label}** (score={likelihood_numeric:.2f})
**Base Likelihood:** {L_base_label} (score={L_base_numeric:.2f})
**Overrides:** {L_override_reason}

### Factor Breakdown
- Threat Actor Capability: {tac_label} (score={tac_score})
- Vulnerability Exploitability: {vuln_label} (score={vuln_score})
- Exposure: {exp_label} (score={exp_score})
- Historical Occurrence: {hist_label} (score={hist_score})

<details>
<summary>Detailed Likelihood Explanation</summary>

{likelihood_basis}

</details>

---

## 4) Impact Analysis
**Final Impact:** **{I_final_label}** (score={I_final_numeric:.2f})
**Base Impact:** {I_base_label} (score={I_base_numeric:.2f})
**Overrides:** {I_override_reason}

### Factor Breakdown
- Safety: {s_label} (score={s_score})
- Availability: {a_label} (score={a_score})
- Confidentiality: {c_label} (score={c_score})
- Integrity: {i_label} (score={i_score})
- Compliance: {comp_label} (score={comp_score})
- Reputation: {rep_label} (score={rep_score})

<details>
<summary>Detailed Impact Explanation</summary>

{impact_basis}

</details>

---

## 5) Overall Risk Rating
- **Risk Rating:** **{rating}**

<details>
<summary>Heatmap Explanation</summary>

{rating_expl}

</details>

---

## 6) Recommended Controls
{controls}

<details>
<summary>Control Rationale</summary>

{controls_rationale}

</details>

---

## 7) Notes for Regulators and Auditors
- Likelihood model incorporates TAC, exploitability, exposure, and historical occurrence.
- Impact model prioritizes safety and compliance.
- Controls derived from NEI 08-09 using RAG.
"""

    return report

# %%
# ============================================================
# 8) PDF Export (ReportLab)
# ============================================================

def export_report_to_pdf(report_text):
    """
    Convert the Markdown report text into a simple PDF and return the file path.
    """
    pdf_path = "OT_Nuclear_Risk_Report.pdf"
    c = canvas.Canvas(pdf_path, pagesize=letter)
    width, height = letter

    y = height - 40
    lines = report_text.split("\n")

    for line in lines:
        wrapped = wrap(line, 110)
        for wline in wrapped:
            c.drawString(40, y, wline)
            y -= 14
            if y < 40:
                c.showPage()
                y = height - 40

    c.save()
    return pdf_path

def generate_pdf_file(report_text):
    """
    Wrapper for Gradio: returns a downloadable PDF file.
    """
    return export_report_to_pdf(report_text)


In [7]:
# %% [markdown]
# # 📘 Section 5 — Main Pipeline, Gradio UI, PDF Download, Launch
# This section:
# - Defines the full assessment pipeline
# - Builds the Gradio UI (ENEC/Nawah enterprise shell)
# - Integrates the PDF export button
# - Launches the application

# %%
# ============================================================
# 9) MAIN FUNCTION TO RUN EVERYTHING (Optimized)
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent
    2. Impact agent
    3. Heatmap risk rating
    4. Control recommendations
    5. Final report generation
    """

    # Step 1: Likelihood
    L_label, L_numeric, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact
    I_label, I_basis, I_details = impact_agent(
        asset_category,
        asset_criticality,
        safety,
        availability,
        confidentiality,
        integrity,
        COMPLIANCE_PATH,
        risk_title,
        risk_causes
    )

    # Step 3: Risk Rating
    R, R_expl = risk_estimator(L_label, I_label, HEATMAP_PATH)

    # Step 4: Controls
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, CONTROL_PATH
    )

    # Step 5: Final Report
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L_label, L_numeric, L_basis, L_details,
        I_label, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    likelihood_md = (
        f"### Likelihood\n"
        f"**Final Likelihood:** {L_label} (score={L_numeric:.2f})\n\n"
        f"{L_basis}"
    )

    impact_md = f"### Impact\n**Final Impact:** {I_label}\n\n{I_basis}"
    risk_md = f"### Risk Rating\n**Rating:** {R}\n\n{R_expl}"
    controls_md = f"### Controls\n{C}\n\n### Rationale\n{C_rat}"

    return (
        likelihood_md,
        impact_md,
        risk_md,
        controls_md,
        report
    )

# %%
# ============================================================
# 10) GRADIO UI — ENEC/Nawah Enterprise Shell (Clean Layout)
# ============================================================

LOGO_URL = "https://github.com/raheelarif86/AI_Training_November25/blob/main/Enec_Logo.png?raw=true"

css = """
/* -----------------------------------------------------------
   GLOBAL FONTS
----------------------------------------------------------- */
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&family=Noto+Sans+Arabic:wght@300;400;600&display=swap');

* {
    font-family: 'Inter', 'Noto Sans Arabic', sans-serif !important;
}

/* -----------------------------------------------------------
   COLOR VARIABLES
----------------------------------------------------------- */
:root {
    --primary-blue: #004C97;
    --secondary-teal: #009CA6;
    --sidebar-bg: #F5F7FA;
    --footer-bg: #002B5C;
    --text-color: #222;
    --bg-color: #ffffff;
}

/* -----------------------------------------------------------
   TOP BANNER
----------------------------------------------------------- */
.top-banner {
    background: linear-gradient(90deg, var(--primary-blue), var(--secondary-teal));
    color: white;
    padding: 18px 24px;
    font-size: 26px;
    font-weight: 700;
}

/* -----------------------------------------------------------
   COLLAPSIBLE SIDEBAR
----------------------------------------------------------- */
.sidebar-container {
    width: 240px;
    background: var(--sidebar-bg);
    border-right: 1px solid #dcdcdc;
    transition: width 0.3s ease;
    overflow: hidden;
    min-height: 100%;
}

.sidebar-collapsed {
    width: 70px !important;
}

.sidebar-logo {
    width: 120px;
    margin: 10px auto;
    display: block;
}

.sidebar-title {
    font-size: 18px;
    font-weight: 600;
    color: var(--primary-blue);
    text-align: center;
}

.sidebar-toggle {
    cursor: pointer;
    padding: 8px;
    text-align: center;
    background: var(--primary-blue);
    color: white;
    font-size: 14px;
}

/* -----------------------------------------------------------
   MAIN CONTENT
----------------------------------------------------------- */
.main-content {
    padding: 20px 30px;
    margin-left: 10px;
}

.gradio-tabs {
    margin-top: 10px !important;
}

/* -----------------------------------------------------------
   FOOTER
----------------------------------------------------------- */
.footer {
    background: var(--footer-bg);
    color: white;
    padding: 12px;
    text-align: center;
    font-size: 13px;
    margin-top: 20px;
}

/* -----------------------------------------------------------
   PRINT-READY PDF STYLING
----------------------------------------------------------- */
@media print {
    body {
        background: white !important;
        color: black !important;
    }
    .sidebar-container, .top-banner, .footer, button, .sidebar-toggle {
        display: none !important;
    }
    .main-content {
        margin: 0;
        padding: 0;
    }
}
"""

with gr.Blocks(css=css, elem_id="app_container") as ui:

    # ---------------- TOP BANNER ----------------
    gr.HTML("""
        <div class="top-banner">
            ENEC / Nawah — OT Nuclear Cyber Risk Assessment Platform
        </div>
    """)

    # ---------------- MAIN LAYOUT ----------------
    with gr.Row():

        # -------- SIDEBAR (collapsible) --------
        with gr.Column(scale=2, min_width=200):
            gr.HTML(f"""
                <div id="sidebar" class="sidebar-container">
                    <div class="sidebar-toggle" onclick="toggleSidebar()">☰</div>
                    <img src="{LOGO_URL}" class="sidebar-logo" />
                    <div class="sidebar-title">Nuclear Cybersecurity</div>
                    <ul style="font-size: 14px; padding: 10px 20px;">
                        <li>OT Risk Assessment</li>
                        <li>Regulatory Alignment</li>
                        <li>NEI 08-09 Controls</li>
                        <li>FANR-REG-08 Compliance</li>
                        <li>Safety-Critical Focus</li>
                    </ul>
                </div>

                <script>
                    function toggleSidebar() {{
                        const sb = document.getElementById('sidebar');
                        sb.classList.toggle('sidebar-collapsed');
                    }}
                </script>
            """)

        # -------- MAIN CONTENT AREA --------
        with gr.Column(scale=10, min_width=900):
            gr.HTML('<div class="main-content">')

            # ---------------- TABS ----------------
            with gr.Tabs():

                # ===== TAB 1: ASSESSMENT =====
                with gr.Tab("Assessment"):

                    with gr.Row():
                        asset_category = gr.Dropdown(
                            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
                            label="Asset Category"
                        )
                        asset_criticality = gr.Dropdown(
                            ["Very High","High","Medium","Low"],
                            label="Asset Criticality"
                        )

                    risk_title = gr.Textbox(label="Risk Title")
                    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

                    with gr.Row():
                        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
                        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

                    with gr.Row():
                        safety_dd = gr.Dropdown(IMPACT, label="Safety Impact")
                        availability_dd = gr.Dropdown(IMPACT, label="Availability Impact")
                        confidentiality_dd = gr.Dropdown(IMPACT, label="Confidentiality Impact")
                        integrity_dd = gr.Dropdown(IMPACT, label="Integrity Impact")

                    run_btn = gr.Button("Run Assessment", variant="primary")

                    likelihood_out = gr.Markdown()
                    impact_out = gr.Markdown()
                    risk_out = gr.Markdown()

                # ===== TAB 2: CONTROLS =====
                with gr.Tab("Controls"):
                    controls_out = gr.Markdown("Run assessment to populate controls.")

                # ===== TAB 3: REPORTS =====
                with gr.Tab("Reports"):
                    report_out = gr.Markdown("Full compiled report will appear here.")

                    # PDF Export Button
                    export_pdf_btn = gr.Button("Export as PDF", variant="secondary")
                    pdf_file_out = gr.File(label="Download PDF")

            # ---------------- PIPELINE BINDING ----------------
            run_btn.click(
                run_full_assessment,
                inputs=[
                    asset_category,
                    asset_criticality,
                    risk_title,
                    risk_causes,
                    threat_actor_capability,
                    exposure,
                    safety_dd,
                    availability_dd,
                    confidentiality_dd,
                    integrity_dd
                ],
                outputs=[
                    likelihood_out,
                    impact_out,
                    risk_out,
                    controls_out,
                    report_out
                ]
            )

            # PDF Export
            export_pdf_btn.click(
                fn=generate_pdf_file,
                inputs=report_out,
                outputs=pdf_file_out
            )

            gr.HTML('</div>')  # close main-content

    # ---------------- FOOTER ----------------
    gr.HTML("""
        <div class="footer">
            © Emirates Nuclear Energy Corporation (ENEC) — Nuclear Cybersecurity Division
            All rights reserved. For internal use only.
        </div>
    """)

# %%
# ============================================================
# 11) LAUNCH APP
# ============================================================

ui.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://1212f7ef47b618d7d1.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [8]:
# %% [markdown]
# # 📘 Section 5 — Main Pipeline, Updated Gradio UI, PDF Download, Launch
# This section:
# - Defines the full assessment pipeline
# - Builds the updated Gradio UI (Assessment + Controls merged)
# - Renames "Reports" → "Report"
# - Integrates the PDF export button
# - Launches the application

# %%
# ============================================================
# 9) MAIN FUNCTION TO RUN EVERYTHING (Optimized)
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent
    2. Impact agent
    3. Heatmap risk rating
    4. Control recommendations
    5. Final report generation
    """

    # Step 1: Likelihood
    L_label, L_numeric, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact
    I_label, I_basis, I_details = impact_agent(
        asset_category,
        asset_criticality,
        safety,
        availability,
        confidentiality,
        integrity,
        COMPLIANCE_PATH,
        risk_title,
        risk_causes
    )

    # Step 3: Risk Rating
    R, R_expl = risk_estimator(L_label, I_label, HEATMAP_PATH)

    # Step 4: Controls
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, CONTROL_PATH
    )

    # Step 5: Final Report
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L_label, L_numeric, L_basis, L_details,
        I_label, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    likelihood_md = (
        f"### Likelihood\n"
        f"**Final Likelihood:** {L_label} (score={L_numeric:.2f})\n\n"
        f"{L_basis}"
    )

    impact_md = f"### Impact\n**Final Impact:** {I_label}\n\n{I_basis}"
    risk_md = f"### Risk Rating\n**Rating:** {R}\n\n{R_expl}"
    controls_md = f"### Controls\n{C}\n\n### Rationale\n{C_rat}"

    return (
        likelihood_md,
        impact_md,
        risk_md,
        controls_md,
        report
    )

# %%
# ============================================================
# 10) UPDATED GRADIO UI — Assessment + Controls merged
# ============================================================

LOGO_URL = "https://github.com/raheelarif86/AI_Training_November25/blob/main/Enec_Logo.png?raw=true"

css = """
/* -----------------------------------------------------------
   GLOBAL FONTS
----------------------------------------------------------- */
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&family=Noto+Sans+Arabic:wght@300;400;600&display=swap');

* {
    font-family: 'Inter', 'Noto Sans Arabic', sans-serif !important;
}

/* -----------------------------------------------------------
   COLOR VARIABLES
----------------------------------------------------------- */
:root {
    --primary-blue: #004C97;
    --secondary-teal: #009CA6;
    --sidebar-bg: #F5F7FA;
    --footer-bg: #002B5C;
    --text-color: #222;
    --bg-color: #ffffff;
}

/* -----------------------------------------------------------
   TOP BANNER
----------------------------------------------------------- */
.top-banner {
    background: linear-gradient(90deg, var(--primary-blue), var(--secondary-teal));
    color: white;
    padding: 18px 24px;
    font-size: 26px;
    font-weight: 700;
}

/* -----------------------------------------------------------
   COLLAPSIBLE SIDEBAR
----------------------------------------------------------- */
.sidebar-container {
    width: 240px;
    background: var(--sidebar-bg);
    border-right: 1px solid #dcdcdc;
    transition: width 0.3s ease;
    overflow: hidden;
    min-height: 100%;
}

.sidebar-collapsed {
    width: 70px !important;
}

.sidebar-logo {
    width: 120px;
    margin: 10px auto;
    display: block;
}

.sidebar-title {
    font-size: 18px;
    font-weight: 600;
    color: var(--primary-blue);
    text-align: center;
}

.sidebar-toggle {
    cursor: pointer;
    padding: 8px;
    text-align: center;
    background: var(--primary-blue);
    color: white;
    font-size: 14px;
}

/* -----------------------------------------------------------
   MAIN CONTENT
----------------------------------------------------------- */
.main-content {
    padding: 20px 30px;
    margin-left: 10px;
}

.gradio-tabs {
    margin-top: 10px !important;
}

/* -----------------------------------------------------------
   FOOTER
----------------------------------------------------------- */
.footer {
    background: var(--footer-bg);
    color: white;
    padding: 12px;
    text-align: center;
    font-size: 13px;
    margin-top: 20px;
}

/* -----------------------------------------------------------
   PRINT-READY PDF STYLING
----------------------------------------------------------- */
@media print {
    body {
        background: white !important;
        color: black !important;
    }
    .sidebar-container, .top-banner, .footer, button, .sidebar-toggle {
        display: none !important;
    }
    .main-content {
        margin: 0;
        padding: 0;
    }
}
"""

with gr.Blocks(css=css, elem_id="app_container") as ui:

    # ---------------- TOP BANNER ----------------
    gr.HTML("""
        <div class="top-banner">
            ENEC / Nawah — OT Nuclear Cyber Risk Assessment Platform
        </div>
    """)

    # ---------------- MAIN LAYOUT ----------------
    with gr.Row():

        # -------- SIDEBAR --------
        with gr.Column(scale=2, min_width=200):
            gr.HTML(f"""
                <div id="sidebar" class="sidebar-container">
                    <div class="sidebar-toggle" onclick="toggleSidebar()">☰</div>
                    <img src="{LOGO_URL}" class="sidebar-logo" />
                    <div class="sidebar-title">Nuclear Cybersecurity</div>
                    <ul style="font-size: 14px; padding: 10px 20px;">
                        <li>OT Risk Assessment</li>
                        <li>Regulatory Alignment</li>
                        <li>NEI 08-09 Controls</li>
                        <li>FANR-REG-08 Compliance</li>
                        <li>Safety-Critical Focus</li>
                    </ul>
                </div>

                <script>
                    function toggleSidebar() {{
                        const sb = document.getElementById('sidebar');
                        sb.classList.toggle('sidebar-collapsed');
                    }}
                </script>
            """)

        # -------- MAIN CONTENT AREA --------
        with gr.Column(scale=10, min_width=900):
            gr.HTML('<div class="main-content">')

            with gr.Tabs():

                # ======================================================
                # SINGLE MERGED TAB: ASSESSMENT (Assessment + Controls)
                # ======================================================
                with gr.Tab("Assessment"):

                    with gr.Row():
                        asset_category = gr.Dropdown(
                            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
                            label="Asset Category"
                        )
                        asset_criticality = gr.Dropdown(
                            ["Very High","High","Medium","Low"],
                            label="Asset Criticality"
                        )

                    risk_title = gr.Textbox(label="Risk Title")
                    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

                    with gr.Row():
                        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
                        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

                    with gr.Row():
                        safety_dd = gr.Dropdown(IMPACT, label="Safety Impact")
                        availability_dd = gr.Dropdown(IMPACT, label="Availability Impact")
                        confidentiality_dd = gr.Dropdown(IMPACT, label="Confidentiality Impact")
                        integrity_dd = gr.Dropdown(IMPACT, label="Integrity Impact")

                    run_btn = gr.Button("Run Assessment", variant="primary")

                    likelihood_out = gr.Markdown()
                    impact_out = gr.Markdown()
                    risk_out = gr.Markdown()
                    controls_out = gr.Markdown()

                # ======================================================
                # TAB 2: REPORT (renamed from Reports)
                # ======================================================
                with gr.Tab("Report"):
                    report_out = gr.Markdown("Full compiled report will appear here.")

                    export_pdf_btn = gr.Button("Export as PDF", variant="secondary")
                    pdf_file_out = gr.File(label="Download PDF")

            # ---------------- PIPELINE BINDING ----------------
            run_btn.click(
                run_full_assessment,
                inputs=[
                    asset_category,
                    asset_criticality,
                    risk_title,
                    risk_causes,
                    threat_actor_capability,
                    exposure,
                    safety_dd,
                    availability_dd,
                    confidentiality_dd,
                    integrity_dd
                ],
                outputs=[
                    likelihood_out,
                    impact_out,
                    risk_out,
                    controls_out,
                    report_out
                ]
            )

            # PDF Export
            export_pdf_btn.click(
                fn=generate_pdf_file,
                inputs=report_out,
                outputs=pdf_file_out
            )

            gr.HTML('</div>')  # close main-content

    # ---------------- FOOTER ----------------
    gr.HTML("""
        <div class="footer">
            © Emirates Nuclear Energy Corporation (ENEC) — Nuclear Cybersecurity Division
            All rights reserved. For internal use only.
        </div>
    """)

# %%
# ============================================================
# 11) LAUNCH APP
# ============================================================

ui.launch(share=True)


IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://2949087016280641c0.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
